# Statistical Inference with BizLensLearn **hypothesis testing, confidence intervals, and statistical comparisons**.Topics Covered:- Confidence intervals (CI) and interpretation- One-sample & two-sample t-tests- ANOVA (comparing 3+ groups)- Correlation analysis- Effect size interpretation

In [ ]:
import subprocess, syssubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bizlens", "scipy", "statsmodels"])import bizlens as bzimport pandas as pdimport numpy as npimport matplotlib.pyplot as plt

## Dataset: A/B Testing Experiment

In [ ]:
# Simulate A/B test: Control vs Treatment groupnp.random.seed(42)control = np.random.normal(loc=100, scale=15, size=50)treatment = np.random.normal(loc=110, scale=15, size=50)df = pd.DataFrame({    'group': ['Control']*50 + ['Treatment']*50,    'metric': list(control) + list(treatment)})print("Experimental Design:")print(f"Control group: n={len(control)}, mean={control.mean():.1f}")print(f"Treatment group: n={len(treatment)}, mean={treatment.mean():.1f}")print(f"Apparent difference: {treatment.mean() - control.mean():.1f}\n")print(df.head())

## 1. Confidence Intervals

In [ ]:
# 95% Confidence interval for control group meanci_control = bz.inference.confidence_interval(control, confidence=0.95)print("\n=== Control Group - 95% Confidence Interval ===")print(f"Point estimate (mean): {ci_control['point_estimate']:.2f}")print(f"95% CI: [{ci_control['lower']:.2f}, {ci_control['upper']:.2f}]")print(f"Margin of error: ±{ci_control['margin_of_error']:.2f}")# Same for treatmentci_treatment = bz.inference.confidence_interval(treatment, confidence=0.95)print("\n=== Treatment Group - 95% Confidence Interval ===")print(f"Point estimate (mean): {ci_treatment['point_estimate']:.2f}")print(f"95% CI: [{ci_treatment['lower']:.2f}, {ci_treatment['upper']:.2f}]")print(f"Margin of error: ±{ci_treatment['margin_of_error']:.2f}")

**Interpretation**: If we repeated this experiment 100 times, ~95 of the confidence intervals would contain the true population mean.

## 2. Hypothesis Testing - Two-Sample t-test

In [ ]:
# Test: Is treatment significantly different from control?# H0: μ_control = μ_treatment# H1: μ_control ≠ μ_treatment (two-tailed)result = bz.inference.two_sample_ttest(    control,    treatment,    equal_var=True,  # Assume equal variances    alternative='two-sided')print("\n=== Two-Sample t-test Results ===")print(f"Test statistic (t): {result['statistic']:.4f}")print(f"p-value: {result['p_value']:.4f}")print(f"Significance level (α): 0.05")if result['p_value'] < 0.05:    print("\n✓ SIGNIFICANT: Reject H0 - Treatment differs from Control (p < 0.05)")else:    print("\n✗ NOT SIGNIFICANT: Fail to reject H0 - No evidence of difference (p ≥ 0.05)")print(f"\nEffect size (Cohen's d): {result.get('cohens_d', 'N/A')}")

## 3. Effect Size Interpretation

In [ ]:
# What does the effect size mean?cohens_d = result.get('cohens_d', 0.5)interpretation = bz.inference.effect_size_interpreter(cohens_d, 'cohens_d')print("\n=== Effect Size Interpretation ===")print(f"Cohen's d: {cohens_d:.3f}")print(f"Interpretation: {interpretation}")

## 4. ANOVA - Comparing Multiple Groups

In [ ]:
# Create a multi-group datasetgroup_a = np.random.normal(100, 15, 40)group_b = np.random.normal(105, 15, 40)group_c = np.random.normal(110, 15, 40)df_multi = pd.DataFrame({    'group': ['A']*40 + ['B']*40 + ['C']*40,    'value': list(group_a) + list(group_b) + list(group_c)})# Run ANOVAanova_result = bz.inference.anova_test(    df_multi,    group_col='group',    value_col='value')print("\n=== One-Way ANOVA Results ===")print(f"F-statistic: {anova_result['f_statistic']:.4f}")print(f"p-value: {anova_result['p_value']:.4f}")if anova_result['p_value'] < 0.05:    print("\n✓ SIGNIFICANT: At least one group differs (p < 0.05)")else:    print("\n✗ NOT SIGNIFICANT: No significant difference between groups (p ≥ 0.05)")

## 5. Correlation Analysis

In [ ]:
# Create two correlated variablesx = np.random.normal(100, 20, 100)y = x + np.random.normal(0, 10, 100)  # y depends on xcorr_result = bz.inference.correlation_test(x, y)print("\n=== Pearson Correlation Test ===")print(f"Correlation coefficient (r): {corr_result['correlation']:.4f}")print(f"p-value: {corr_result['p_value']:.4f}")print(f"R-squared (R²): {corr_result['r_squared']:.4f}")print(f"\nInterpretation:")print(f"- Variables have {'STRONG' if abs(corr_result['correlation']) > 0.7 else 'MODERATE'} correlation")print(f"- {corr_result['r_squared']*100:.1f}% of variation in y is explained by x")

## 6. Visualization of Results

In [ ]:
import matplotlib.pyplot as pltfig, axes = plt.subplots(2, 2, figsize=(14, 10))# Distribution comparisonaxes[0,0].hist(control, bins=15, alpha=0.6, label='Control', color='blue')axes[0,0].hist(treatment, bins=15, alpha=0.6, label='Treatment', color='orange')axes[0,0].axvline(control.mean(), color='blue', linestyle='--', linewidth=2, label=f'Control μ={control.mean():.1f}')axes[0,0].axvline(treatment.mean(), color='orange', linestyle='--', linewidth=2, label=f'Treatment μ={treatment.mean():.1f}')axes[0,0].set_title('Distribution Comparison', fontsize=12, fontweight='bold')axes[0,0].set_xlabel('Metric Value')axes[0,0].set_ylabel('Frequency')axes[0,0].legend()# Box plotdf.boxplot(column='metric', by='group', ax=axes[0,1])axes[0,1].set_title('Box Plot Comparison', fontsize=12, fontweight='bold')axes[0,1].set_xlabel('Group')axes[0,1].set_ylabel('Metric')# Confidence intervalsgroups = ['Control', 'Treatment']means = [ci_control['point_estimate'], ci_treatment['point_estimate']]errors = [ci_control['margin_of_error'], ci_treatment['margin_of_error']]axes[1,0].errorbar(groups, means, yerr=errors, fmt='o-', markersize=10, capsize=5, capthick=2)axes[1,0].set_title('95% Confidence Intervals', fontsize=12, fontweight='bold')axes[1,0].set_ylabel('Mean ± 95% CI')axes[1,0].grid(axis='y', alpha=0.3)# Correlation scatteraxes[1,1].scatter(x, y, alpha=0.6)z = np.polyfit(x, y, 1)p = np.poly1d(z)axes[1,1].plot(x, p(x), "r--", linewidth=2, label=f'r={corr_result["correlation"]:.3f}')axes[1,1].set_title('Correlation: X vs Y', fontsize=12, fontweight='bold')axes[1,1].set_xlabel('X')axes[1,1].set_ylabel('Y')axes[1,1].legend()plt.tight_layout()plt.show()

## Summary of Statistical Concepts| Concept | Purpose | p-value < 0.05 | Interpretation ||---------|---------|----------------|-----------------|| **Confidence Interval** | Estimate population parameter | N/A | If CI excludes 0, suggest significance || **t-test** | Compare 2 groups | Significant difference | Reject H0 - groups differ || **ANOVA** | Compare 3+ groups | Significant difference | At least one group differs || **Correlation** | Relationship strength | Significant relationship | Variables are associated |## Key Takeaways✓ **p-value < 0.05** = statistically significant at 95% confidence✓ **Effect size** matters: statistical significance ≠ practical significance✓ **Confidence intervals** give a range, not just a point estimate✓ **Assumptions matter**: t-tests assume normality, ANOVA assumes equal variances✓ **Correlation ≠ Causation**: X and Y are related, but X may not cause Y